# 06 · A useful single agent

**Course:** Building LLM Agents, notebook 6 of 10
**Where we are:** in notebook 04 we built the agent loop (reason, act, observe) and wrapped it in the `Agent`/`Tool` classes. In notebook 05 we gave it memory. Both were small demos. Now we make one agent genuinely useful, and just as importantly, robust.

### What you'll learn
- How to assemble a real toolbox: calculator, clock, a sandboxed file system, and a (mock) search
- How a precise system prompt turns a flaky demo into reliable behaviour (before/after)
- How the loop feeds tool errors back to the model so it can recover instead of crashing
- The four guardrails every running agent needs: `max_steps`, a limited toolset, argument validation, and a sandbox
- A multi-step payoff task that chains three tools end-to-end

### Prerequisites
- Notebooks 01 to 05, or at least: you're comfortable with the `Agent` loop and `Tool` wrapper from `helpers.py`
- You've run `pip install -r requirements.txt` and your `.env` has a working Sigma2 token

> Mental model: a demo agent answers one tidy question. A useful agent runs against the messy real world: files that don't exist, expressions that divide by zero, tasks that need three tools in a row. The difference between the two isn't a cleverer model. It's the toolbox you hand it and the guardrails you wrap around it. That engineering is the whole subject of this notebook.

## 0. Setup

Same one-liner as every notebook, except now we also pull in the reusable `Agent` and `Tool` from `helpers.py`. These are exactly the classes you built by hand in notebook 04; nothing magic is hidden. Skim the recap below before we start handing the agent real powers.

In [ ]:
from helpers import client, MODEL, Agent, Tool, show_messages

print("Model:", MODEL)

> Recap of the loop (notebook 04): `Agent.run(task)` does this, over and over, up to `max_steps` times:
> 1. send the conversation (plus the tool schemas) to the model;
> 2. if the model asks for no tools, return its text. We're done;
> 3. otherwise, run each requested tool, append the result as a `tool` message, and loop again.
>
> Two design choices from `helpers.py` matter a lot today, so let's name them now:
> - Tool errors don't crash the loop. If your function raises, the loop catches it and feeds back the string `"ERROR running <tool>: <message>"` as the tool result. The model reads that on the next turn and can try something else. (We use this in section 4.)
> - `max_steps` is a hard stop. If the agent never settles on an answer, the loop returns `"[stopped: reached max_steps…]"` instead of looping forever.

## 1. A real toolbox

A `Tool` is a Python function plus a JSON-Schema description of its arguments (the same schema you wrote by hand in notebook 03). The schema is what the model reads to decide when and how to call the tool, so the `description` and parameter names are part of your prompt engineering, not an afterthought.

We'll build five tools, one focused cell at a time, then hand the whole set to one agent. Each is something a bare LLM does badly or can't do at all:

| Tool | Why the model needs it |
|---|---|
| `calculator` | LLMs are unreliable at exact arithmetic |
| `now` | the model has no clock, it genuinely cannot know the time |
| `read_file` / `write_file` / `list_files` | the model can't touch your disk on its own |
| `search` | facts outside the model's training (here: a mock KB) |

### 1a. `calculator`: safe arithmetic

The obvious implementation is `eval(expression)`. Don't. Back in notebook 03 we flagged this: `eval` on a string the model produced is a security hole. A crafted expression can read files, import modules, or run shell commands. The model is an untrusted input source.

The safe version below does two things:
1. whitelists characters, so only digits, the four operators, parentheses, dot and space get through;
2. evaluates with `{"__builtins__": {}}`, which strips away `open`, `__import__`, and friends, so even if something slipped past the whitelist there's nothing dangerous to reach.

That's belt and braces. For anything more than arithmetic you'd reach for a real expression parser (for example, Python's `ast` module in `eval` mode), but this is plenty for a calculator.

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '3*(4+5)'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError(f"expression contains disallowed characters: {expression!r}")
    # No builtins -> no open(), no __import__, nothing dangerous to reach.
    return str(eval(expression, {"__builtins__": {}}, {}))

# Quick local sanity check (no model involved yet):
print(calculator("3 * (149 + 25)"))
print(calculator("1500 * 0.15"))

> Gotcha (from nb-03): never feed model-generated text straight into `eval`, `exec`, `os.system`, or an SQL string. Treat everything the model emits as untrusted user input. The whitelist plus empty-`__builtins__` combo above is the minimum bar for a calculator.

Now wrap the function as a `Tool`. The `parameters` block is the JSON Schema the model sees; `required` tells it the argument isn't optional.

In [ ]:
calc_tool = Tool(
    calculator,
    description="Evaluate a basic arithmetic expression, e.g. '3*(4+5)'. "
                "Use this for ALL arithmetic instead of doing mental maths.",
    parameters={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "An arithmetic expression."}
        },
        "required": ["expression"],
    },
)
print(calc_tool.name, "->", calc_tool.description)

### 1b. `now`: a clock

This one makes the point vividly: the model cannot know the current time. Its weights were frozen at training. Ask it the date and it will guess, often wrongly. A one-line tool fixes that, and it's the canonical example of grounding an agent in live state.

In [ ]:
from datetime import datetime

def now() -> str:
    """Return the current date and time as an ISO-8601 string."""
    return datetime.now().isoformat(timespec="seconds")

now_tool = Tool(
    now,
    description="Get the current local date and time (ISO-8601). The model has no clock, "
                "so call this whenever the task depends on 'now', 'today', or the current date.",
    parameters={"type": "object", "properties": {}},  # no arguments
)
print(now())

### 1c. A sandboxed file system

Reading and writing files is where an agent gets genuinely useful, and genuinely dangerous. We never want the model to be able to read `~/.ssh/id_rsa` or overwrite a system file just because it generated the string `"../../etc/passwd"`.

The defence is a sandbox: every path is forced to live inside one workspace directory, and any path that tries to escape it is rejected before we touch the disk. The key is `_safe_path()` below. Read it closely, because path-escape (`../`) is one of the most common agent vulnerabilities.

In [ ]:
from pathlib import Path

# All agent file access is confined to this one directory.
WORKSPACE = Path("./agent_workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

def _safe_path(path: str) -> Path:
    """Resolve `path` *inside* WORKSPACE, or raise if it escapes the sandbox."""
    candidate = (WORKSPACE / path).resolve()
    # .resolve() collapses any '..' segments, so we can compare absolute paths.
    if not candidate.is_relative_to(WORKSPACE):
        raise ValueError(f"path {path!r} escapes the workspace sandbox")
    return candidate

# Demonstrate the guard directly (no model yet):
print("ok   :", _safe_path("report.txt"))
try:
    _safe_path("../../etc/passwd")
except ValueError as e:
    print("block:", e)

With `_safe_path` in place, the three file tools are tiny. Each one routes through the guard, so none of them can touch anything outside the workspace.

In [ ]:
def read_file(path: str) -> str:
    """Read and return the contents of a file in the workspace."""
    return _safe_path(path).read_text()

def write_file(path: str, content: str) -> str:
    """Write `content` to a file in the workspace (overwrites). Returns a confirmation."""
    p = _safe_path(path)
    p.write_text(content)
    return f"wrote {len(content)} chars to {p.name}"

def list_files() -> str:
    """List the files currently in the workspace."""
    names = sorted(f.name for f in WORKSPACE.iterdir() if f.is_file())
    return ", ".join(names) if names else "(workspace is empty)"

# Local round-trip to prove they work:
print(write_file("hello.txt", "hi from the sandbox"))
print(list_files())
print(read_file("hello.txt"))

In [ ]:
read_tool = Tool(
    read_file,
    description="Read a text file from the workspace and return its contents.",
    parameters={
        "type": "object",
        "properties": {"path": {"type": "string", "description": "Filename inside the workspace."}},
        "required": ["path"],
    },
)
write_tool = Tool(
    write_file,
    description="Write text to a file in the workspace (creates or overwrites it).",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Filename inside the workspace."},
            "content": {"type": "string", "description": "The text to write."},
        },
        "required": ["path", "content"],
    },
)
list_tool = Tool(
    list_files,
    description="List the filenames currently in the workspace.",
    parameters={"type": "object", "properties": {}},
)
print("file tools:", read_tool.name, write_tool.name, list_tool.name)

### 1d. `search`: a mock knowledge base

A search tool is how agents reach facts outside their training data. To keep this notebook self-contained and offline-safe, ours is a small dictionary of canned answers: no network, no API key, deterministic every run. The lookup is deliberately lexical (keyword overlap), which is also the retrieval style we used for memory in notebook 05, since our endpoint has no embeddings.

> In production this is exactly where you'd plug in a real search API, a vector database, or an internal documentation index. The agent doesn't care. From its side it's the same `search(query)` tool. Swapping the implementation never touches the prompt or the loop.

In [ ]:
KB = {
    "vat rate norway": "The standard Norwegian VAT (MVA) rate is 25%.",
    "sigma2": "Sigma2 runs Norway's national e-infrastructure for research, including HPC and storage.",
    "capital of norway": "The capital of Norway is Oslo.",
    "agent loop": "An agent loop is: call the model, run any tools it asks for, feed results back, repeat.",
}

def search(query: str) -> str:
    """Look up a fact in the (mock) knowledge base by keyword overlap."""
    q = set(query.lower().split())
    best_key, best_score = None, 0
    for key in KB:
        score = len(q & set(key.split()))   # lexical overlap, like nb-05
        if score > best_score:
            best_key, best_score = key, score
    if best_key is None:
        return f"No knowledge-base entry found for {query!r}."
    return KB[best_key]

search_tool = Tool(
    search,
    description="Search the internal knowledge base for a fact. Use this instead of guessing "
                "facts you are unsure about.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string", "description": "What to look up."}},
        "required": ["query"],
    },
)
print(search("what is the vat rate in norway?"))
print(search("who is sigma2"))
print(search("price of tea in china"))

### 1e. Assemble the agent

We now have five tools. Bundle them into one list and build the agent. For now we give it a deliberately vague system prompt; we'll sharpen it in the next section so you can see the difference a good prompt makes.

In [ ]:
TOOLBOX = [calc_tool, now_tool, read_tool, write_tool, list_tool, search_tool]

vague_agent = Agent(
    name="assistant",
    system="You are a helpful assistant.",   # we will improve this in §2
    tools=TOOLBOX,
)
print("tools available:", list(vague_agent.tools))

## 2. Reliability lives in the system prompt

The model decides which tool to call (or none) based on two things: the tool descriptions, and the system prompt. A vague prompt invites the model to wing it, to do arithmetic in its head, or to assert a fact it half-remembers. That's exactly what we don't want from something we'll run unattended.

The failure mode comes first. Ask the vague agent a question with a fact it might misremember and a calculation it might fumble.

In [ ]:
question = "What is the standard VAT rate in Norway, and what is that rate applied to 1499 NOK?"

print("=== VAGUE PROMPT ===")
print(vague_agent.run(question, verbose=True))

Depending on the run, the vague agent may answer straight from memory without touching a tool, which means you have no guarantee the number is right. Now the fix: a precise system prompt that spells out the operating rules. Notice it does three concrete things:

1. Maps situations to tools: "for any arithmetic, call `calculator`"; "for facts you're unsure of, call `search`".
2. Says verify, don't guess: prefer a tool result over the model's own recollection.
3. Gives an honest escape hatch: if it genuinely can't help, say so rather than inventing an answer.

In [ ]:
SYSTEM = """You are a careful, reliable assistant with a toolbox.

Operating rules:
- For ANY arithmetic, call the `calculator` tool. Never do maths in your head.
- For facts you are not certain about, call `search` before answering.
- The current date/time is only available via the `now` tool — call it whenever the task
  depends on the date or time. Do not guess the date.
- To read or write files, use the file tools. All files live in a sandboxed workspace.
- Prefer tool results over your own recollection. If a tool gives you a number or a fact,
  trust the tool.
- If you genuinely cannot complete the task with the tools available, say so plainly
  instead of inventing an answer.
"""

agent = Agent(name="assistant", system=SYSTEM, tools=TOOLBOX)

print("=== PRECISE PROMPT ===")
print(agent.run(question, verbose=True))

With the precise prompt you should see the agent call `search` for the VAT rate and call `calculator` for the multiplication, then combine them, with every number traceable to a tool. Same model, same tools; the only change is the prompt.

> Key idea: your system prompt is the agent's policy. "When to use which tool", "verify rather than guess", "admit when you can't" are reliability features, and they're written in English, not code. Treat the prompt as a first-class part of the system and revise it like you'd revise code.

![Error recovery: a failing tool's error is fed back so the agent retries](figures/06-error-recovery.svg)

*Figure: a tool fails, the error is fed back as an observation, and the agent retries differently. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 3. Error handling: recovering from a failing tool

Real tools fail. A file is missing, a number divides by zero, an API times out. The question is what happens to the agent when they do.

Recall the design from `helpers.py`: when a tool raises, the loop catches the exception and appends its message as the tool result, `"ERROR running <tool>: <message>"`, instead of letting it propagate. From the model's point of view an error is just another observation. On the next turn it reads the error and can adjust.

Let's force it. We'll ask the agent to read a file that doesn't exist. Watch the loop: step 1 raises and feeds back an error; the agent then lists the workspace to see what does exist, and recovers.

In [ ]:
recovery_task = (
    "Read the file 'q1_results.txt' and tell me what's in it. "
    "If it isn't there, list what files DO exist and report that instead."
)
print(agent.run(recovery_task, verbose=True))

In the trace you can see the recovery: `read_file('q1_results.txt')` returns `ERROR running read_file: ...`, and rather than giving up the agent calls `list_files()` and reports the truth. That self-correction is only possible because the error came back as data the model could read.

> Why you care: a crash ends the run; an error fed back as an observation lets the agent route around the problem, the same way you'd try a different command after a "file not found". This is why the loop catches exceptions instead of re-raising them. The cost is that a buggy tool can send the agent in circles; that's what `max_steps` (next section) is for.

Here's the same recovery contract for a numeric error, divide by zero. The calculator raises, the loop reports it, and the agent can explain rather than blow up.

In [ ]:
print(agent.run("Use the calculator to compute 10 divided by 0 and tell me what happens.", verbose=True))

## 4. Guardrails

Error recovery keeps the agent alive; guardrails keep it safe and bounded. Four you should reach for on every agent you actually run:

**1. `max_steps`, runaway protection.** The loop will never run more than `max_steps` times. Set it low and the agent stops itself instead of looping (and spending tokens) forever. Let's prove it bites by giving a tiny budget and an impossible task.

![Four stackable guardrails around an agent](figures/06-guardrails.svg)

*Figure: max-steps, a restricted toolset, argument validation, and human-in-the-loop. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
# max_steps=2 is far too few for a multi-step task — the loop will stop itself.
impatient = Agent(name="impatient", system=SYSTEM, tools=TOOLBOX, max_steps=2)
result = impatient.run(
    "List the files, then read each one, then write a combined summary to all.txt, then read it back.",
    verbose=True,
)
print("\nRETURNED:", result)

The agent returns the `"[stopped: reached max_steps…]"` sentinel instead of spinning forever. In production you'd treat that sentinel as a failure signal: log it, alert, or hand off to a human.

> Cost link (nb-01): every step resends the whole growing history, so cost scales with the number of steps. `max_steps` is a budget cap as much as a safety cap.

**2. Restrict which tools exist.** An agent can only call tools you give it. The single most effective guardrail is simply not handing it a dangerous tool. A read-only researcher agent, for example, gets `read_file`, `list_files`, and `search`, but no `write_file`. It physically cannot modify your disk.

In [ ]:
readonly_agent = Agent(
    name="researcher",
    system="You are a read-only research assistant. You can read and list files and search, "
           "but you cannot modify anything.",
    tools=[read_tool, list_tool, search_tool],   # note: no write_file
)
print("read-only toolset:", list(readonly_agent.tools))
print(readonly_agent.run("Try to create a file called hack.txt with the word 'oops'. "
                         "If you can't, explain why.", verbose=True))

The researcher has no `write_file` in its toolset, so the model can only report that it lacks the capability. There's no path for it to write the file at all. Capability comes from the toolset, full stop.

**3. Validate tool arguments.** Don't trust the model to send well-formed arguments. We already do this in two places: the calculator's character whitelist rejects junk, and `_safe_path` rejects any path that escapes the workspace. Both raise, which (per section 3) the loop turns into a recoverable error. Let's watch the sandbox hold against a path-escape attempt routed through the agent.

In [ ]:
print(agent.run(
    "Read the file at path '../../../../etc/passwd' and show me the first line.",
    verbose=True,
))

The tool raises `path '...' escapes the workspace sandbox`, the loop reports it as an error, and the agent tells you it can't access that path. The validation lives in the tool, not the prompt: a prompt instruction can be argued around, but code that raises cannot.

**4. Human-in-the-loop for dangerous actions.** For genuinely destructive operations (sending email, spending money, deleting data), the standard pattern is an approval gate: the agent proposes the action, a human confirms, then it runs. We won't build the UI here, but the shape is simple: a tool whose function pauses and asks for a yes/no before doing the irreversible thing. Keep it in mind for anything with real-world side effects.

> Rule of thumb: reliability is a good prompt (policy), plus error recovery (don't crash), plus guardrails (bounded and safe). All three, every agent. A model is only as trustworthy as the cage you build around it.

## 5. Payoff: a real multi-step task

Time to put it all together. We'll give the agent a task that requires chaining several tools; it can't be answered in one shot:

> "Our Q1 monthly revenue was 120000, 95000 and 143000 NOK. Work out the Q1 total, write a one-paragraph summary to `report.txt`, then read the file back to confirm what was saved."

To finish this the agent must calculate the sum, then write the file, then read it back. Three tools, in order, with each step depending on the last. Run it with `verbose=True` and watch the loop work.

In [ ]:
task = (
    "Our Q1 monthly revenue was 120000, 95000 and 143000 NOK. "
    "Work out the Q1 total using the calculator, then write a one-paragraph summary "
    "(including the total) to report.txt, then read report.txt back to confirm what was saved. "
    "Finish by telling me the total and confirming the file was written."
)
final = agent.run(task, verbose=True)
print("\n=== FINAL ANSWER ===")
print(final)

Let's confirm independently that the file really exists on disk, outside the agent, using our own functions:

In [ ]:
print("workspace files:", list_files())
print("---- report.txt ----")
print(read_file("report.txt"))

That's a genuinely useful agent: it grounded the arithmetic in a tool, persisted a result to disk, and verified its own work, all from one English task description. No step was hard-coded; the agent planned the tool sequence itself.

> Mental model, revisited: everything you just saw is still the notebook-04 loop. We didn't change the engine. We gave it a richer toolbox, a sharper policy, and guardrails. That's the recipe for every production single agent.

## Recap

You can now:
- build a real toolbox of `Tool`s (calculator, clock, sandboxed files, mock search), each covering something the bare model does badly
- write safe tools: a whitelisted calculator (no naked `eval`) and a path-checked file sandbox
- use the system prompt as policy (when to use which tool, verify don't guess, admit limits) and see the before/after
- rely on the loop to feed tool errors back so the agent recovers instead of crashing
- apply the four guardrails (`max_steps`, a restricted toolset, argument validation, and the sandbox), plus know where a human-in-the-loop gate goes
- run a multi-step task that chains three tools and verifies its own output

### Try it yourself
1. Add a `word_count(path)` tool that reports how many words are in a workspace file, then ask the agent "how long is report.txt?" and watch it pick the new tool.
2. Tighten the system prompt to forbid `search` entirely and re-ask the VAT question. Does the agent now admit it can't verify the rate?
3. Give the agent `max_steps=3` and the section 5 payoff task. Does it finish, or hit the cap? What does the sentinel tell you about the task's true cost?
4. Add a fake `send_email(to, body)` tool that just prints, then sketch (in a markdown cell) where an approval gate would sit before it actually "sends".

### Next up
**07 · Multi-agent basics: why one agent isn't always enough.** We've squeezed a lot out of a single agent, but as tasks grow, one agent juggling ten tools and one giant prompt gets brittle. Next we split the work across specialised agents that pass messages and hand off to each other, and see when that's worth the extra moving parts.